# Zadanie 1: dostęp do LLM i mini-wyszukiwarka

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/01-llm-embeddingi/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-01`, w załączniku ten notebook **z wynikami komórek**, bez kluczy API.

**Termin:** przed zajęciami 2.

Części:
1. Dostęp do modelu i wywołanie z wymuszonym formatem odpowiedzi.
2. Koszt w tokenach: polski i angielski.
3. Wyszukiwarka na własnych dokumentach i analiza błędu.

## 0. Konfiguracja

In [1]:
import tiktoken
from openai.types.beta.threads import message
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
%pip install -q openai tiktoken numpy scikit-learn matplotlib seaborn python-dotenv anthropic

Note: you may need to restart the kernel to use updated packages.


Dostawcę wybiera się zmienną `LLM_PROVIDER` (plik `.env` albo w Colabie: ikona klucza → *Secrets*). Instrukcja konfiguracji dostępu: [00-dostep-do-llm](../00-dostep-do-llm/README.md).

| `LLM_PROVIDER` | Wymagania |
|---|---|
| `azure` | `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY` i deploymenty modeli |
| `openai` | `OPENAI_API_KEY` |
| `deepseek` | `DEEPSEEK_API_KEY`; tanie API, bez embeddingów |
| `ollama` | [Ollama](https://ollama.com) lokalnie albo w Colabie na darmowym GPU (komórka niżej), modele `gemma4:e4b`, `gemma3:1b`, `embeddinggemma` |
| `proxy` | uruchomione [proxy dla subskrypcji](../tools/teg-proxy/README.md) (Copilot, Claude, Codex); bez embeddingów |

Wszyscy dostawcy udostępniają API zgodne z OpenAI, więc dalszy kod jest wspólny. Gdy dostawca nie liczy embeddingów, służy do tego `EMBED_PROVIDER` (domyślnie `ollama`).

### Ollama w Colabie (darmowe GPU)

Colab daje za darmo kartę graficzną NVIDIA T4, na której działa Ollama. Najpierw *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*, potem komórka poniżej (odkomentowana). Na początek wystarczy mały `gemma3:1b`; większy `gemma4:e4b` zajmuje ok. 10 GB. Bez niego w *Secrets* ustawia się `OLLAMA_CHAT=gemma3:1b`.

In [ ]:
# Colab z GPU: odkomentuj i uruchom raz na sesję
# !nvidia-smi                                   # czy GPU jest włączone
# !sudo apt-get -qq install -y zstd pciutils > /dev/null
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess
# subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# !ollama pull gemma3:1b
# !ollama pull embeddinggemma
# !ollama pull gemma4:e4b                       # opcjonalnie, ~10 GB

In [3]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "EMBED_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_CHAT_DEPLOYMENT", "AZURE_OPENAI_CLASSIC_DEPLOYMENT",
              "AZURE_OPENAI_EMBED_DEPLOYMENT", "DEEPSEEK_API_KEY", "ANTHROPIC_API_KEY"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

# chat: model rozumujący, classic: model bez rozumowania, embed: model embeddingów.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini", "embed": "text-embedding-3-small"},
    "azure":    {
        "chat": os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT") or os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5-nano"),
        "classic": os.getenv("AZURE_OPENAI_CLASSIC_DEPLOYMENT") or os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini"),
        "embed": os.getenv("AZURE_OPENAI_EMBED_DEPLOYMENT", "text-embedding-3-small"),
    },
    "deepseek": {"chat": "deepseek-reasoner", "classic": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma3:1b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

czat: azure (gpt-4.1-mini, klasyczny: gpt-4.1-mini) | embeddingi: azure (text-embedding-3-small)


## Część 1. Wywołanie modelu

1. Wywołanie `CHAT` z własnym system promptem, który wymusza format odpowiedzi (np. JSON z ustalonymi polami).
2. Wypisanie treści odpowiedzi, `finish_reason` i `usage`.
3. Sprawdzenie w kodzie, czy odpowiedź ma oczekiwany format (np. `json.loads`), i wypisanie wyniku sprawdzenia.

In [4]:
import json

SYSTEM = "Zwracaj mi wyniki tylko w formacie JSON ma być poprawny, bez md. Odpowiadaj wyłącznie w JSON z polami `answer` i `confidence"
USER = "Czym jest embedding, jedno zdanie."

response = client.chat.completions.create(
    model = CHAT,
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": USER}
    ]
)

content = response.choices[0].message.content

try:
    data = json.loads(content)
    ok_format_flag = (
        isinstance(data, dict)
        and "answer" in data
        and "confidence" in data
        and isinstance(data["answer"], str))

except json.JSONDecodeError:
    data = None
    ok_format_flag = False

print("Format:", ok_format_flag)
if data is None:
    print("Data:", data)

print(content)
print("finish reason:", response.choices[0].finish_reason)
print("usage:", response.usage)



Format: True
{
  "answer": "Embedding to reprezentacja danych, takich jak słowa czy obrazy, w postaci wektorów liczbowych, które zachowują ich semantyczne właściwości.",
  "confidence": 0.99
}
finish reason: stop
usage: CompletionUsage(completion_tokens=53, prompt_tokens=59, total_tokens=112, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), latency_checkpoint={'engine_tbt_ms': 5, 'engine_ttft_ms': 21, 'engine_ttlt_ms': 309, 'pre_inference_ms': 90, 'service_tbt_ms': 5, 'service_ttft_ms': 403, 'service_ttlt_ms': 687, 'user_visible_ttft_ms': 313})


## Część 2. Koszt w tokenach

1. Akapit tekstu po polsku (min. 300 znaków) i jego tłumaczenie na angielski.
2. Liczba tokenów obu wersji według tokenizera `o200k_base` (`tiktoken`).
3. Wysłanie obu wersji do modelu z tym samym poleceniem i porównanie `usage.prompt_tokens`.
4. W komórce markdown: o ile procent droższa jest wersja polska i z czego to wynika.

In [5]:
# TODO: liczenie tokenów i porównanie usage
import tiktoken


def compare_tokens(a:float, b:float, label_A:str , label_B: str) -> None:
    if a == b:
        print(f"{label_A} are the same {label_B}")
        return
    if a > b:
        bigger, smaller = a, b
        bigger_label, smaller_label = label_A, label_B
    else:
        bigger, smaller = b, a
        bigger_label, smaller_label = label_B, label_A

    percentage = (bigger - smaller)/smaller*100
    print(f"{bigger_label} is {percentage}% more expense")

enc = tiktoken.get_encoding("o200k_base")
text_pl = "Bilbo Baggins urządza ogromne przyjęcie z okazji swoich 111. urodzin. Podczas przemowy niespodziewanie znika dzięki Pierścieniowi, po czym opuszcza Shire i zostawia swój majątek Frodowi."
text_eng = "Bilbo Baggins throws a huge party for his 111th birthday. During his speech, he suddenly disappears using the Ring, then leaves the Shire behind and passes his home and possessions to Frodo."

instructions_pl = "Powiedz kto jest autorem tego tekstu."
instructions_eng = "Tell me who's the author of this text."

enc_pl = enc.encode(text_pl)
enc_eng = enc.encode(text_eng)

#print(enc_pl)
#print(enc_eng)

pieces = [enc.decode([i]) for i in enc_pl]
#print(f"{len(text_pl)} znakow -> {len(enc_pl):2d} tokenow | {pieces}")
#print("----")
pieces2 = [enc.decode([j]) for j in enc_eng]
#print(f"{len(text_eng)} chars -> {len(enc_eng):2d} tokens | {pieces2}")

# Promt responses
response_pl = client.chat.completions.create(
    model = CHAT,
    messages = [
        {"role": "system", "content": instructions_pl},
        {"role": "user", "content": text_pl}
    ]
)
response_eng = client.chat.completions.create(
    model = CHAT,
    messages = [
        {"role": "system", "content": instructions_eng},
        {"role": "user", "content": text_eng}
    ]
)

content_pl = response_pl.choices[0].message.content
content_eng = response_eng.choices[0].message.content
tkn_cnt_pl = response_pl.usage.prompt_tokens
tkn_cnt_eng = response_eng.usage.prompt_tokens
print("PL usage:", tkn_cnt_pl)
print("ENG usage:", tkn_cnt_eng)

compare_tokens(tkn_cnt_pl, tkn_cnt_eng, "Polish", "English")



PL usage: 78
ENG usage: 63
Polish is 23.809523809523807% more expense


*Odpowiedź:*
Polish is 23.809523809523807% more expense

## Część 3. Mini-wyszukiwarka

1. Co najmniej **10 krótkich dokumentów** (1–2 zdania) z jednej, wybranej dziedziny.
2. Embeddingi dokumentów (Azure: `client.embeddings`, deployment `EMBED`) i funkcja `search(query, k=3)` oparta na cosinusie.
3. **3 pytania**: dwa z poprawną odpowiedzią wyszukiwarki i jedno z błędną.
4. W komórce markdown: przyczyna błędu (2–4 zdania).

In [6]:
import numpy as np

docs = [
    "SK hynix to południowokoreański producent półprzewodników pamięci, jeden z największych na świecie w segmencie DRAM i NAND.",
    "Siedziba główna firmy znajduje się w Icheon w prowincji Gyeonggi w Korei Południowej.",
    "Firma powstała w 1983 roku jako Hyundai Electronics; później działała jako Hynix Semiconductor, a od 2012 roku jest częścią grupy SK.",
    "SK hynix produkuje m.in. pamięci DRAM do komputerów, serwerów i urządzeń mobilnych.",
    "Firma rozwija i dostarcza pamięci NAND flash oraz rozwiązania pamięci masowych, w tym pod marką Solidigm po przejęciu działu pamięci NAND Intela.",
    "SK hynix jest jednym z głównych dostawców pamięci HBM (High Bandwidth Memory) wykorzystywanych w akceleratorach AI i GPU.",
    "Do głównych konkurentów SK hynix w rynku pamięci należą Samsung Electronics oraz Micron Technology.",
    "Produkcja półprzewodników SK hynix opiera się na zaawansowanych fabrykach (ośrodki w Korei, m.in. okolice Cheongju i Yongin).",
    "Firma notuje się na Koreańskiej Giełdzie Papierów Wartościowych pod tickerem 000660.",
    "SK hynix inwestuje w kolejne generacje litografii i skalowania procesów (np. EUV), aby zwiększać gęstość i wydajność układów pamięci.",
]

def _vectors(texts: list[str]) -> np.ndarray:
    rows = client.embeddings.create(model=EMBED, input=texts).data
    return np.array([r.embedding for r in rows])

D = _vectors(docs)

def search(query: str, k: int = 3) -> list[tuple[float, str]]:
    q = _vectors([query])[0]
    scores = (D @ q) / (np.linalg.norm(D, axis=1) * np.linalg.norm(q))
    best = np.argsort(-scores)[:k]
    return [(round(float(scores[i]), 3), docs[i]) for i in best]


embedding OK (azure/text-embedding-3-small), wymiar: 1536
10 dokumentów, wymiar embeddingu: 1536 (azure/text-embedding-3-small)


In [7]:
questions = [
    ("trafne", "Jaką pamięć SK hynix dostarcza do akceleratorów AI i GPU?"),
    ("trafne", "W jakim mieście w Korei Południowej znajduje się siedziba główna SK hynix?"),
    ("błędne", "Ile wynosi dzienna kapitalizacja giełdowa Apple w dolarach?"),
]

for label, q in questions:
    print(f"[{label}] Q: {q}")
    for score, doc in search(q, k=3):
        print(f"   {score:.3f}  {doc}")
    print()

[trafne] Q: Jaką pamięć SK hynix dostarcza do akceleratorów AI i GPU?
   0.813  SK hynix jest jednym z głównych dostawców pamięci HBM (High Bandwidth Memory) wykorzystywanych w akceleratorach AI i GPU.
   0.681  SK hynix inwestuje w kolejne generacje litografii i skalowania procesów (np. EUV), aby zwiększać gęstość i wydajność układów pamięci.
   0.666  SK hynix to południowokoreański producent półprzewodników pamięci, jeden z największych na świecie w segmencie DRAM i NAND.

[trafne] Q: W jakim mieście w Korei Południowej znajduje się siedziba główna SK hynix?
   0.691  Produkcja półprzewodników SK hynix opiera się na zaawansowanych fabrykach (ośrodki w Korei, m.in. okolice Cheongju i Yongin).
   0.689  SK hynix to południowokoreański producent półprzewodników pamięci, jeden z największych na świecie w segmencie DRAM i NAND.
   0.652  Siedziba główna firmy znajduje się w Icheon w prowincji Gyeonggi w Korei Południowej.

[błędne] Q: Ile wynosi dzienna kapitalizacja giełdowa Apple w dol

*Wyjaśnienie błędu:*

Pytanie o kapitalizację Apple nie ma odpowiedzi w korpusie (10 faktów tylko o SK hynix). Wyszukiwarka i tak zwraca trzy najbliższe wektory — np. zdanie o tickerze 000660 lub o konkurentach — bo embedding łączy ogólne słowa giełdowe i nazwy firm, a nie „czy w tekście jest dokładna odpowiedź”. To nie jest błąd API, tylko ograniczenie samego podobieństwa cosinusowego bez sprawdzania faktów.